# Training curves
Load `campaign_analysis.h5` (scp'd from cluster) and plot all summary metrics over rounds.

In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

H5_PATH = "campaign_analysis.h5"  # adjust path after scp
ROLLING = 10  # rounds for rolling mean

plt.rcParams.update({'font.size': 10, 'axes.grid': True, 'grid.alpha': 0.3})

f = h5py.File(H5_PATH, 'r')
s = f['summary']
rounds = s['round'][:]
print(f"Campaign: {f.attrs.get('campaign_name', '?')}  N={f.attrs.get('N')}  rounds={len(rounds)}")

In [ ]:
def rolling(arr, w):
    out = np.full_like(arr, np.nan)
    for i in range(len(arr)):
        chunk = arr[max(0, i-w+1):i+1]
        valid = chunk[np.isfinite(chunk)]
        if valid.size:
            out[i] = valid.mean()
    return out

def panel(ax, x, y, title, ylabel=None, hline=None, color='steelblue'):
    valid = np.isfinite(y)
    ax.plot(x[valid], y[valid], '.', ms=2, alpha=0.4, color=color)
    rm = rolling(y, ROLLING)
    ax.plot(x[np.isfinite(rm)], rm[np.isfinite(rm)], '-', lw=1.5, color=color)
    if hline is not None:
        ax.axhline(hline, color='k', lw=0.8, ls='--', alpha=0.6)
    ax.set_title(title, fontsize=9)
    ax.set_xlabel('round')
    if ylabel:
        ax.set_ylabel(ylabel)

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(14, 14))
ax = axes.ravel()

panel(ax[0], rounds, s['mean_reward'][:],      'Training mean reward',     hline=0)
panel(ax[1], rounds, s['eval_dphi'][:],        'eval ⟨φ − φ_null⟩',       hline=0)
panel(ax[2], rounds, s['eval_success'][:],     'Eval success rate')

# sigma_policy: exp(log_std) — pulled from policy group if available
if 'policy' in f:
    pr = f['policy']['rounds'][:]
    ls = f['policy']['log_std'][:]  # (R, 2)
    ax[3].plot(pr, np.exp(ls[:, 0]), '.-', ms=2, lw=1, label='aP')
    ax[3].plot(pr, np.exp(ls[:, 1]), '.-', ms=2, lw=1, label='aS')
    ax[3].set_title('Policy std (exploration)'); ax[3].set_xlabel('round')
    ax[3].legend(fontsize=8)
elif 'sigma_policy' in s:
    panel(ax[3], rounds, s['sigma_policy'][:], 'Policy σ')

panel(ax[4], rounds, s['Bbar'][:],             'Mean bulk modulus B')
panel(ax[5], rounds, s['Gbar'][:],             'Mean shear modulus G')
panel(ax[6], rounds, s['dzbar'][:],            'Mean Δz (excess coordination)')
panel(ax[7], rounds, s['rattler_frac'][:],     'Rattler fraction')
panel(ax[8], rounds, s['omega_star'][:],       'ω* (5th pct eigenfreq)')

# actions on one panel
ax9 = ax[9]
for key, label, c in [('mean_absaP','|aP|','tab:blue'), ('mean_absaS','|aS|','tab:orange'), ('mean_absgamma','|γ|','tab:green')]:
    if key in s:
        v = s[key][:]
        ax9.plot(rounds[np.isfinite(v)], v[np.isfinite(v)], '.', ms=2, alpha=0.4, label=label)
        rm = rolling(v, ROLLING)
        ax9.plot(rounds[np.isfinite(rm)], rm[np.isfinite(rm)], '-', lw=1.5, label=f'{label} (avg)')
ax9.set_title('Mean absolute actions'); ax9.set_xlabel('round'); ax9.legend(fontsize=7)

panel(ax[10], rounds, s['shear_stable_frac'][:], 'Shear-stable fraction')

if 'wall_seconds' in s:
    panel(ax[11], rounds, s['wall_seconds'][:], 'Wall time per round (s)')
else:
    ax[11].set_visible(False)

name = f.attrs.get('campaign_name', '')
fig.suptitle(f'Training curves — {name}', fontsize=12, y=1.01)
fig.tight_layout()
fig.savefig('training_curves.pdf', bbox_inches='tight')
plt.show()
print('Saved training_curves.pdf')